# TS Forecasting V10: Exact 2604 + Blend Candidates

Focus: maximize leaderboard score.

This notebook is a near-exact reproduction of `lb-2604-20-seeds` with 20-seed LGBM per horizon, plus automatic export of 4 candidate submissions:
1. exact model with clipping
2. exact model without clipping
3. blend 85/15
4. blend 75/25


In [1]:
import gc
import os
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

T0 = time.time()

TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.parquet'
    TEST_PATH = 'test.parquet'

VAL_THRESHOLD = 3500
HORIZONS = [1, 3, 10, 25]

LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.015,
    'n_estimators': 5000,
    'num_leaves': 90,
    'min_child_samples': 200,
    'feature_fraction': 0.65,
    'bagging_fraction': 0.75,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 10.0,
    'verbosity': -1,
    'n_jobs': -1,
}

SEEDS_20 = [42, 2024, 12345, 99, 420, 777, 1337, 2025, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

# Optional external reference file for blending.
# If None or missing, notebook will blend clip/no-clip variants.
REFERENCE_SUB_PATH = '/kaggle/input/datasets/yaroslavkholmirzayev/blend70a30b/submission_v9_blend70A30B.csv'

print('TRAIN_PATH:', TRAIN_PATH)
print('TEST_PATH :', TEST_PATH)
print('VAL_THRESHOLD:', VAL_THRESHOLD)
print('HORIZONS:', HORIZONS)
print('SEEDS:', len(SEEDS_20))


TRAIN_PATH: /kaggle/input/competitions/ts-forecasting/train.parquet
TEST_PATH : /kaggle/input/competitions/ts-forecasting/test.parquet
VAL_THRESHOLD: 3500
HORIZONS: [1, 3, 10, 25]
SEEDS: 20


In [2]:
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))


def weighted_rmse_score(y_target, y_pred, w) -> float:
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)

    denom = np.sum(w * (y_target ** 2))
    if denom <= 0:
        return 0.0

    ratio = np.sum(w * ((y_target - y_pred) ** 2)) / denom
    return float(np.sqrt(1.0 - _clip01(ratio)))


In [3]:
print('Computing encoding stats (fit split only)...')

tmp = pd.read_parquet(TRAIN_PATH, columns=['sub_category', 'sub_code', 'y_target', 'ts_index'])
train_only = tmp[tmp.ts_index <= VAL_THRESHOLD]

TRAIN_STATS = {
    'sub_category': train_only.groupby('sub_category')['y_target'].mean().to_dict(),
    'sub_code': train_only.groupby('sub_code')['y_target'].mean().to_dict(),
    'global_mean': float(train_only['y_target'].mean()),
}

del tmp, train_only
gc.collect()

print('Stats ready.')


Computing encoding stats (fit split only)...
Stats ready.


## Exact Feature Pipeline (2604 style)


In [4]:
def build_features_exact(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()
    group_cols = ['code', 'sub_code', 'sub_category', 'horizon']

    # Interactions
    df['d_al_am'] = df['feature_al'] - df['feature_am']
    df['r_al_am'] = df['feature_al'] / (df['feature_am'] + 1e-7)
    df['d_cg_by'] = df['feature_cg'] - df['feature_by']

    if 'feature_s' in df.columns:
        df['s_al_prod'] = df['feature_s'] * df['feature_al']
        df['s_am_prod'] = df['feature_s'] * df['feature_am']
        df['s_cg_prod'] = df['feature_s'] * df['feature_cg']

    # Encodings from fit period only
    for c in ['sub_category', 'sub_code']:
        df[c + '_enc'] = df[c].map(TRAIN_STATS[c]).fillna(TRAIN_STATS['global_mean'])

    # Cross-sectional normalization
    for col in ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am']:
        if col in df.columns:
            g = df.groupby('ts_index')[col]
            df[col + '_cs'] = (df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)

    # Cyclical time
    df['t_sin'] = np.sin(2 * np.pi * df['ts_index'] / 100.0)
    df['t_cos'] = np.cos(2 * np.pi * df['ts_index'] / 100.0)

    # Time-series engine
    df = df.sort_values(group_cols + ['ts_index'])

    target_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by']
    if 'feature_s' in df.columns:
        target_cols.append('feature_s')

    grouped = df.groupby(group_cols, sort=False)

    for col in target_cols:
        for lag in [1, 3, 5, 10, 25]:
            df[f'{col}_lag{lag}'] = grouped[col].shift(lag)

        for w in [5, 10, 20]:
            df[f'{col}_roll_mean_{w}'] = grouped[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{col}_roll_std_{w}'] = grouped[col].transform(lambda x: x.rolling(w, min_periods=1).std())

        df[f'{col}_ewm_10'] = grouped[col].transform(lambda x: x.ewm(span=10, min_periods=1).mean())

        df[f'{col}_diff1'] = grouped[col].diff(1)
        df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)

    return df.fillna(0.0)


def get_feature_columns(df: pd.DataFrame):
    exclude = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude]


## Train Per Horizon (20 Seeds + Retrain-on-All)


In [5]:
def solve_horizon_exact(horizon: int):
    t_h = time.time()
    print()
    print('=' * 70)
    print(f'HORIZON {horizon}')
    print('=' * 70)

    tr = pd.read_parquet(TRAIN_PATH).query('horizon == @horizon')
    te = pd.read_parquet(TEST_PATH).query('horizon == @horizon')

    print(f'Data: train={len(tr):,}, test={len(te):,}')

    tr = build_features_exact(tr)
    te = build_features_exact(te)

    feats = get_feature_columns(tr)
    for c in feats:
        if c not in te.columns:
            te[c] = 0.0

    print(f'Features: {len(feats)}')

    fit_m = tr.ts_index <= VAL_THRESHOLD
    val_m = tr.ts_index > VAL_THRESHOLD

    X_fit = tr.loc[fit_m, feats]
    y_fit = tr.loc[fit_m, 'y_target']
    w_fit = tr.loc[fit_m, 'weight']

    X_val = tr.loc[val_m, feats]
    y_val = tr.loc[val_m, 'y_target']
    w_val = tr.loc[val_m, 'weight']

    X_all = tr[feats]
    y_all = tr['y_target']
    w_all = tr['weight']

    val_pred = np.zeros(len(y_val), dtype=np.float64)
    test_pred = np.zeros(len(te), dtype=np.float64)
    best_iters = []

    for i, seed in enumerate(SEEDS_20, 1):
        if i == 1 or i % 5 == 1:
            print(f'  Seed {i}/{len(SEEDS_20)} ...')

        mdl = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        mdl.fit(
            X_fit,
            y_fit,
            sample_weight=w_fit,
            eval_set=[(X_val, y_val)],
            eval_sample_weight=[w_val],
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )

        bi = mdl.best_iteration_ if mdl.best_iteration_ is not None else LGB_PARAMS['n_estimators']
        bi = max(int(bi), 20)
        best_iters.append(bi)

        val_pred += mdl.predict(X_val) / len(SEEDS_20)

        mdl_full = lgb.LGBMRegressor(**{**LGB_PARAMS, 'n_estimators': bi}, random_state=seed)
        mdl_full.fit(X_all, y_all, sample_weight=w_all)
        test_pred += mdl_full.predict(te[feats]) / len(SEEDS_20)

        del mdl, mdl_full
        gc.collect()

    score = weighted_rmse_score(y_val.values, val_pred, w_val.values)

    q_low, q_high = np.quantile(y_fit.values, [CLIP_Q_LOW, CLIP_Q_HIGH])
    test_pred_clip = np.clip(test_pred, q_low, q_high)

    print(f'Local score={score:.6f} | avg_best_iter={np.mean(best_iters):.1f} | elapsed={(time.time()-t_h)/60:.1f} min')

    out = {
        'horizon': horizon,
        'id_test': te['id'].values,
        'test_pred_raw': test_pred,
        'test_pred_clip': test_pred_clip,
        'id_val': tr.loc[val_m, 'id'].values,
        'y_val': y_val.values,
        'w_val': w_val.values,
        'val_pred': val_pred,
        'score_local': score,
    }

    del tr, te, X_fit, y_fit, w_fit, X_val, y_val, w_val, X_all, y_all, w_all
    gc.collect()

    return out


In [6]:
all_outputs = []

for h in HORIZONS:
    all_outputs.append(solve_horizon_exact(h))

# Build main submissions
sub_raw_parts = []
sub_clip_parts = []
oof_parts = []

for out in all_outputs:
    sub_raw_parts.append(pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_raw']}))
    sub_clip_parts.append(pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_clip']}))
    oof_parts.append(pd.DataFrame({
        'id': out['id_val'],
        'y_true': out['y_val'],
        'y_pred': out['val_pred'],
        'w': out['w_val'],
        'horizon': out['horizon'],
    }))

sub_raw = pd.concat(sub_raw_parts, ignore_index=True)
sub_clip = pd.concat(sub_clip_parts, ignore_index=True)
oof = pd.concat(oof_parts, ignore_index=True)

agg_local = weighted_rmse_score(oof['y_true'].values, oof['y_pred'].values, oof['w'].values)

sub_clip.to_csv('submission_v10_exact_2604_clip005_995.csv', index=False)
sub_raw.to_csv('submission_v10_exact_2604_no_clip.csv', index=False)
oof.to_csv('oof_v10_exact_2604.csv', index=False)

print()
print('=' * 70)
print('PER-HORIZON LOCAL SCORES')
for out in sorted(all_outputs, key=lambda d: d['horizon']):
    print(f"h={out['horizon']:>2}: {out['score_local']:.6f}")
print('-' * 70)
print(f'Aggregate local score: {agg_local:.6f}')
print('Saved: submission_v10_exact_2604_clip005_995.csv')
print('Saved: submission_v10_exact_2604_no_clip.csv')
print('Saved: oof_v10_exact_2604.csv')
print('=' * 70)



HORIZON 1
Data: train=1,394,653, test=379,617
Features: 171
  Seed 1/20 ...
  Seed 6/20 ...
  Seed 11/20 ...
  Seed 16/20 ...
Local score=0.082151 | avg_best_iter=522.2 | elapsed=130.5 min

HORIZON 3
Data: train=1,385,816, test=376,558
Features: 171
  Seed 1/20 ...
  Seed 6/20 ...
  Seed 11/20 ...
  Seed 16/20 ...
Local score=0.140782 | avg_best_iter=1069.2 | elapsed=223.5 min

HORIZON 10
Data: train=1,337,236, test=362,057
Features: 171
  Seed 1/20 ...
  Seed 6/20 ...
  Seed 11/20 ...
  Seed 16/20 ...
Local score=0.222175 | avg_best_iter=805.7 | elapsed=175.8 min

HORIZON 25
Data: train=1,219,709, test=328,875
Features: 171
  Seed 1/20 ...
  Seed 6/20 ...
  Seed 11/20 ...
  Seed 16/20 ...
Local score=0.273106 | avg_best_iter=667.6 | elapsed=145.3 min

PER-HORIZON LOCAL SCORES
h= 1: 0.082151
h= 3: 0.140782
h=10: 0.222175
h=25: 0.273106
----------------------------------------------------------------------
Aggregate local score: 0.235895
Saved: submission_v10_exact_2604_clip005_995.csv

## Build 2 Blend Candidates

Priority blend source:
- If `REFERENCE_SUB_PATH` exists -> blend with that file (e.g., your best `blend70A30B`).
- Otherwise -> blend `clip` with `no_clip`.


In [7]:
blend_base = sub_clip.copy()
base_name = 'v10clip'

ref_path = REFERENCE_SUB_PATH
if ref_path is not None and os.path.exists(ref_path):
    print('Using external reference for blending:', ref_path)
    ref = pd.read_csv(ref_path, usecols=['id', 'prediction']).rename(columns={'prediction': 'pred_ref'})
    blend_base = blend_base.merge(ref, on='id', how='inner')
    blend_base = blend_base.rename(columns={'prediction': 'pred_v10'})
    p_main = blend_base['pred_v10'].values
    p_ref = blend_base['pred_ref'].values
    base_name = 'v10_vs_ref'
else:
    print('External reference not provided/found. Blending clip vs no-clip.')
    tmp = sub_clip.merge(sub_raw, on='id', suffixes=('_clip', '_raw'))
    blend_base = tmp.rename(columns={'prediction_clip': 'pred_v10', 'prediction_raw': 'pred_ref'})
    p_main = blend_base['pred_v10'].values
    p_ref = blend_base['pred_ref'].values
    base_name = 'v10clip_vs_v10raw'

blend_85 = pd.DataFrame({
    'id': blend_base['id'].values,
    'prediction': 0.85 * p_main + 0.15 * p_ref,
})
blend_75 = pd.DataFrame({
    'id': blend_base['id'].values,
    'prediction': 0.75 * p_main + 0.25 * p_ref,
})

name_85 = f'submission_v10_blend85_15_{base_name}.csv'
name_75 = f'submission_v10_blend75_25_{base_name}.csv'

blend_85.to_csv(name_85, index=False)
blend_75.to_csv(name_75, index=False)

print('Saved:', name_85)
print('Saved:', name_75)


Using external reference for blending: /kaggle/input/datasets/yaroslavkholmirzayev/blend70a30b/submission_v9_blend70A30B.csv
Saved: submission_v10_blend85_15_v10_vs_ref.csv
Saved: submission_v10_blend75_25_v10_vs_ref.csv


## Submit Priority

If external reference was used and is your strongest prior file:
1. `submission_v10_exact_2604_clip005_995.csv`
2. `submission_v10_blend85_15_*.csv`
3. `submission_v10_blend75_25_*.csv`
4. `submission_v10_exact_2604_no_clip.csv`

If no external reference was used, keep same order but treat blend files as fallback around clip/no-clip.
